# SNAP Edge Case Analysis

Deep dive into the policy boundaries that trip up LLMs:
- Gross vs. net income thresholds
- Deduction stacking (earned income + standard + shelter)
- Elderly/disabled gross income waiver
- BBCE states vs. strict asset-test states
- Categorical eligibility overrides

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from govsynth.sources.us.snap import SNAPSource, BBCE_STATES, STRICT_ASSET_TEST_STATES
from govsynth.profiles.us_household import USHouseholdProfile
from govsynth.generators.snap_eligibility import SNAPEligibilityGenerator

print('Ready.')

Ready.


## 1. Gross Income Boundary — 3-Person Household, Virginia

In [2]:
source = SNAPSource(fiscal_year=2026, state='VA')
t = source.thresholds()
limits_3 = t.by_household_size(3)

print(f'3-person gross limit (130% FPL): ${limits_3.gross_monthly:,.2f}/month')
print(f'3-person net limit  (100% FPL): ${limits_3.net_monthly:,.2f}/month')
print()

offsets = [('5% below', -0.05), ('1% below', -0.01), ('AT LIMIT', 0.0), ('1% above', 0.01), ('5% above', 0.05)]
print(f'{"Label":<12} {"Gross Income":<18} {"Eligible?"}')
print('-' * 45)
for label, pct in offsets:
    profile = USHouseholdProfile.at_threshold(
        program='snap', threshold='gross_income_limit',
        state='VA', household_size=3, fiscal_year=2026,
        offset_pct=pct, seed=42,
    )
    eligible, reason = source.is_eligible(
        household_size=profile.household_size,
        gross_income=profile.monthly_gross_income,
        net_income=profile.monthly_net_income,
        liquid_assets=profile.liquid_assets,
    )
    print(f'{label:<12} ${profile.monthly_gross_income:>10,.2f}       {"ELIGIBLE" if eligible else "INELIGIBLE"}')


3-person gross limit (130% FPL): $2,888.00/month
3-person net limit  (100% FPL): $2,221.00/month

Label        Gross Income       Eligible?
---------------------------------------------
5% below     $  2,743.60       ELIGIBLE
1% below     $  2,859.12       ELIGIBLE
AT LIMIT     $  2,888.00       ELIGIBLE
1% above     $  2,916.88       INELIGIBLE
5% above     $  3,032.40       INELIGIBLE


## 2. Deduction Stacking — Net Income Edge Cases

The net income test is what trips models up. It requires applying earned income deduction, standard deduction, and optionally a shelter deduction. Let's show the mechanics for a household right at the net income limit.

In [ ]:
from govsynth.sources.us.snap import get_standard_deduction

hh_size = 4
limits_4 = t.by_household_size(4)
std_ded = get_standard_deduction(hh_size)

# Profile exactly at the net income limit
profile = USHouseholdProfile.at_threshold(
    program='snap', threshold='net_income_limit',
    state='VA', household_size=hh_size, fiscal_year=2026,
    offset_pct=0.0, seed=42,
)

earned_ded = (profile.earned_income or profile.monthly_gross_income) * 0.20
net_manual = profile.monthly_gross_income - earned_ded - std_ded
net_official = source.calculate_net_income(
    gross_income=profile.monthly_gross_income,
    household_size=hh_size,
    earned_income=profile.earned_income,
)

print(f'4-person net income limit: ${limits_4.net_monthly:,.2f}/month')
print()
print(f'Gross income:              ${profile.monthly_gross_income:>10,.2f}')
print(f'- 20% earned deduction:    ${earned_ded:>10,.2f}   (7 CFR 273.9(c)(1))')
print(f'- Standard deduction:      ${std_ded:>10,.2f}   (7 CFR 273.9(c)(2))')
print(f'= Net income:              ${net_manual:>10,.2f}')
print()
print(f'Official net (source):     ${net_official:>10,.2f}')
print(f'Limit:                     ${limits_4.net_monthly:>10,.2f}')
print(f'Eligible:                  {"YES" if net_official <= limits_4.net_monthly else "NO"}')


## 3. Elderly/Disabled Household — Gross Income Waiver

7 CFR 273.9(a)(1): Households with an elderly (60+) or disabled member are **exempt from the gross income test** but must still pass the net income test. This is a common LLM failure mode.

In [ ]:
# Household with income 20% ABOVE gross limit — would fail for normal household
# but is elderly, so gross test is waived
hh_size = 2
limits_2 = t.by_household_size(2)
gross_above = round(limits_2.gross_monthly * 1.20, 2)
net_below = round(limits_2.net_monthly * 0.80, 2)

# Normal household — ineligible
elig_normal, reason_normal = source.is_eligible(
    household_size=hh_size,
    gross_income=gross_above,
    net_income=net_below,
    liquid_assets=500,
    has_elderly_or_disabled=False,
)

# Elderly household — eligible (gross waived, net passes)
elig_elderly, reason_elderly = source.is_eligible(
    household_size=hh_size,
    gross_income=gross_above,
    net_income=net_below,
    liquid_assets=500,
    has_elderly_or_disabled=True,
)

print(f'2-person gross limit: ${limits_2.gross_monthly:,.2f} | net limit: ${limits_2.net_monthly:,.2f}')
print(f'Test household gross: ${gross_above:,.2f} (20% above limit) | net: ${net_below:,.2f}')
print()
print(f'Without elderly/disabled flag: {"ELIGIBLE" if elig_normal else "INELIGIBLE"}')
print(f'  Reason: {reason_normal}')
print()
print(f'With elderly/disabled flag:    {"ELIGIBLE" if elig_elderly else "INELIGIBLE"}')
print(f'  Reason: {reason_elderly}')
print()
print('Note: asset limit for elderly/disabled is $4,500 (not $3,000) per 7 CFR 273.8(b)(2)')


## 4. BBCE vs. Strict Asset Test — Virginia vs. Texas

Broad-Based Categorical Eligibility (BBCE) states waive the asset test for most households. Texas uses the strict federal asset test ($3,000 general). Same income, different outcomes based purely on state.

In [ ]:
source_va = SNAPSource(fiscal_year=2026, state='VA')  # BBCE
source_tx = SNAPSource(fiscal_year=2026, state='TX')  # Strict asset test

for state, src in [('VA (BBCE)', source_va), ('TX (strict)', source_tx)]:
    t_state = src.thresholds()
    asset_display = 'waived (BBCE)' if t_state.asset_limit_general is None else f'${t_state.asset_limit_general:,}'
    print(f'{state}: asset_limit_general = {asset_display}')

print()
# Household with $4,000 in assets, income well below limit
hh_size = 3
limits_3_va = source_va.thresholds().by_household_size(3)
gross = round(limits_3_va.gross_monthly * 0.70, 2)
assets = 4000

print(f'Test case: {hh_size}-person HH, gross=${gross:,.2f}, assets=${assets:,}')
print()
for state, src in [('VA (BBCE)', source_va), ('TX (strict)', source_tx)]:
    elig, reason = src.is_eligible(
        household_size=hh_size,
        gross_income=gross,
        liquid_assets=assets,
    )
    print(f'  {state}: {"ELIGIBLE" if elig else "INELIGIBLE"} — {reason[:80]}')


## 5. Generate Edge-Saturated Cases and Inspect Difficulty Distribution

In [ ]:
from collections import Counter

# Generate a larger batch to see difficulty spread
generator = SNAPEligibilityGenerator(fiscal_year=2026, state='VA')
cases = generator.generate(n=50, seed=42)

print(f'Generated: {len(cases)} cases')
print()

diff = Counter(c.difficulty.value for c in cases)
print('Difficulty distribution:')
for d, n in sorted(diff.items()):
    pct = n / len(cases) * 100
    bar = '█' * int(pct / 2)
    print(f'  {d:<12} {n:>3}  {pct:>5.1f}%  {bar}')

print()
print('Threshold types in cases:')
types = Counter(c.metadata.get('profile_strategy', 'unknown') for c in cases)
for t_type, n in sorted(types.items()):
    print(f'  {t_type:<30} {n}')


## 6. Spot Check: Inspect a Hard Ineligible Case's Rationale Trace

In [ ]:
hard_inelig = [c for c in cases if c.difficulty.value == 'hard' and c.expected_outcome == 'ineligible']
if hard_inelig:
    case = hard_inelig[0]
    print(f'Case: {case.case_id}')
    print(f'Outcome: {case.expected_outcome.upper()}')
    print(f'Scenario: {case.scenario.summary}')
    print()
    print('Rationale trace:')
    print(case.rationale_trace.to_plain_text())
else:
    print('No hard ineligible cases in this batch — try a larger n or different seed.')


---

## 7. Special-Population Edge Cases

The six cases below are where LLMs fail most consistently — not because they get the income arithmetic wrong, but because they apply the wrong rule entirely. Each is generated by a dedicated builder that constructs a full rationale trace citing the exact CFR section.

| # | Edge Case | CFR | What models get wrong |
|---|---|---|---|
| 7.1 | Homeless shelter deduction | 273.9(c)(6) | Applies excess shelter formula instead of flat $198.99 |
| 7.2 | Student exclusion | 273.5(a),(b) | Skips student status check; income-eligible student passes |
| 7.3 | Boarder income proration | 273.1(b)(7) | Counts full board payment instead of profit only |
| 7.4 | Migrant income averaging | 273.10(c)(3) | Uses current-month snapshot instead of averaged income |
| 7.5 | Mixed immigration status | 273.4(c)(3) | Prorates income or includes ineligible member in HH size |
| 7.6 | Categorical eligibility (TANF/SSI) | 273.2(j)(2), 273.11(c) | Runs income test after categorical eligibility is established |

### 7.1 Homeless Shelter Deduction — 7 CFR 273.9(c)(6)

Homeless households receive a **flat $198.99/month deduction** in place of the excess shelter deduction. These two deductions are mutually exclusive — the homeless deduction is not an uncapped version of excess shelter.

The common model error: applying the standard excess shelter formula `(shelter_costs - 50% of net income)` to a homeless household instead of substituting the flat value.

In [ ]:
import random

gen = SNAPEligibilityGenerator(fiscal_year=2026, state='VA')
rng = random.Random(42)
case = gen._build_homeless_case(rng)

t = gen.source.thresholds()
homeless_ded = t.extra['homeless_shelter_deduction']

print(f'Case: {case.case_id}')
print(f'Outcome: {case.expected_outcome.upper()}')
print(f'Scenario: {case.scenario.summary}')
print()
print('Rationale trace:')
for step in case.rationale_trace.steps:
    print(f'  Step {step.step_number}: [{step.rule_applied}] {step.title}')
    print(f'    {step.computation}')
    print(f'    → {step.result}')
    print()
print(f'Conclusion: {case.rationale_trace.conclusion}')
print()
print(f'Key: flat homeless deduction = ${homeless_ded:.2f} (from data/thresholds/snap_fy2026.json)')
print(f'     NOT the excess shelter formula (shelter_costs - 50% of net income)')

### 7.2 Student Exclusion — 7 CFR 273.5(a),(b)

A student enrolled **at least half-time** at an institution of higher education is ineligible unless they meet one of the exceptions in 273.5(b): working 20+ hours/week, single parent with a child under 6, TANF recipient, or work-study participant.

The check fires **before the income test** — income level is irrelevant. A student with income well below the gross limit is still ineligible if no exception applies.

In [ ]:
rng = random.Random(42)
case = gen._build_student_case(rng)

t = gen.source.thresholds()
limits = t.by_household_size(case.scenario.household_size)

print(f'Case: {case.case_id}')
print(f'Outcome: {case.expected_outcome.upper()}')
print(f'Gross income:  ${case.scenario.monthly_gross_income:,.2f}')
print(f'Gross limit:   ${limits.gross_monthly:,.2f}  ← income is BELOW the limit')
print(f'Income below limit? {case.scenario.monthly_gross_income < limits.gross_monthly}')
print()
print('Rationale trace:')
for step in case.rationale_trace.steps:
    print(f'  Step {step.step_number}: [{step.rule_applied}] {step.title}')
    print(f'    {step.computation}')
    if step.note:
        print(f'    Note: {step.note}')
    print(f'    → {step.result}')
    print()
print(f'Conclusion: {case.rationale_trace.conclusion}')

### 7.3 Boarder Income Proration — 7 CFR 273.1(b)(7)

When a household takes in a boarder (someone who pays for room and board), only the **profit** counts as income — not the full payment. Profit = board payment received minus the actual cost of providing food and housing.

Example: $800/month received, $600 in actual costs → $200 countable income (not $800).

In [ ]:
rng = random.Random(42)
case = gen._build_boarder_case(rng)
ctx = case.scenario.additional_context

print(f'Case: {case.case_id}')
print(f'Outcome: {case.expected_outcome.upper()}')
print()
print(f'Board payment received:  ${ctx["board_payment_total"]:>8,.2f}')
print(f'Actual cost to provide:  ${ctx["board_cost"]:>8,.2f}')
print(f'Profit (countable):      ${ctx["boarder_income"]:>8,.2f}  ← only this counts')
print(f'Other wages:             ${case.scenario.monthly_gross_income - ctx["boarder_income"]:>8,.2f}')
print(f'Total countable income:  ${case.scenario.monthly_gross_income:>8,.2f}')
print()
print('Rationale trace:')
for step in case.rationale_trace.steps:
    print(f'  Step {step.step_number}: [{step.rule_applied}] {step.title}')
    print(f'    {step.computation}')
    print(f'    → {step.result}')
    print()

### 7.4 Migrant/Seasonal Worker Income Averaging — 7 CFR 273.10(c)(3)

For seasonal or migrant workers, income is averaged over the **work period**, not taken as a current-month snapshot. A worker who earns $12,000 over 4 months is counted at $3,000/month even if currently between jobs.

Models commonly use the current month's income (zero between seasons) or incorrectly annualize the amount.

In [ ]:
rng = random.Random(42)
case = gen._build_migrant_case(rng)
ctx = case.scenario.additional_context

print(f'Case: {case.case_id}')
print(f'Outcome: {case.expected_outcome.upper()}')
print()
print(f'Seasonal total earnings:  ${ctx["seasonal_total"]:>10,.2f}')
print(f'Work period:              {ctx["work_months"]} months')
print(f'Averaged monthly income:  ${case.scenario.monthly_gross_income:>10,.2f}  (total ÷ months)')
print()
print('Rationale trace:')
for step in case.rationale_trace.steps:
    print(f'  Step {step.step_number}: [{step.rule_applied}] {step.title}')
    print(f'    {step.computation}')
    print(f'    → {step.result}')
    print()

### 7.5 Mixed Immigration Status — 7 CFR 273.4(c)(3)

In a mixed-status household, ineligible (non-qualified alien) members are **excluded from household size** for limit lookup, but their income counts **in full**. This is NOT income proration — that's a different rule for sponsored noncitizens (7 CFR 273.11(c)(3)).

Example: A 4-person household with 1 ineligible member uses **3-person income limits**, but all 4 members' income is counted at 100%.

In [ ]:
rng = random.Random(42)
case = gen._build_mixed_immigration_case(rng)
ctx = case.scenario.additional_context

total = case.scenario.household_size
ineligible = ctx['ineligible_member_count']
eligible = ctx['eligible_member_count']

t = gen.source.thresholds()
limits_full = t.by_household_size(total)
limits_reduced = t.by_household_size(eligible)

print(f'Case: {case.case_id}')
print(f'Outcome: {case.expected_outcome.upper()}')
print()
print(f'Total household members:    {total}')
print(f'Ineligible (non-qualified): {ineligible}')
print(f'Eligible members:           {eligible}  ← used for limit lookup')
print()
print(f'Gross income (full):        ${case.scenario.monthly_gross_income:,.2f}  (all members counted at 100%)')
print(f'{total}-person gross limit:       ${limits_full.gross_monthly:,.2f}  (would apply if no exclusion)')
print(f'{eligible}-person gross limit:       ${limits_reduced.gross_monthly:,.2f}  ← actual limit applied')
print()
print('Rationale trace:')
for step in case.rationale_trace.steps:
    print(f'  Step {step.step_number}: [{step.rule_applied}] {step.title}')
    print(f'    {step.computation}')
    print(f'    → {step.result}')
    print()

### 7.6 Categorical Eligibility (TANF/SSI) — 7 CFR 273.2(j)(2), 273.11(c)

Households receiving TANF cash assistance are categorically eligible for SNAP — the income test is **skipped entirely**. SSI recipients are categorically eligible under 7 CFR 273.11(c). This means a household with income above the 130% FPL gross limit is still **ELIGIBLE** if a member receives TANF or SSI.

The most common model error: running the income test anyway and returning INELIGIBLE for the above-limit household.

In [ ]:
rng = random.Random(42)
case = gen._build_categorical_eligibility_case(rng)
ctx = case.scenario.additional_context

t = gen.source.thresholds()
limits = t.by_household_size(case.scenario.household_size)

print(f'Case: {case.case_id}')
print(f'Outcome: {case.expected_outcome.upper()}')
print()
print(f'Gross income:     ${case.scenario.monthly_gross_income:,.2f}')
print(f'Gross limit:      ${limits.gross_monthly:,.2f}  ← income is ABOVE limit')
print(f'Above limit by:   ${case.scenario.monthly_gross_income - limits.gross_monthly:,.2f}')
print(f'TANF/SSI:         {ctx["tanf_or_ssi_recipient"]}  → income test skipped entirely')
print()
print('Rationale trace:')
for step in case.rationale_trace.steps:
    print(f'  Step {step.step_number}: [{step.rule_applied}] {step.title}')
    print(f'    {step.computation}')
    print(f'    → {step.result}')
    print()
print(f'Conclusion: {case.rationale_trace.conclusion}')

### 7.7 All Special-Population Cases in One Batch

Generate a full batch and filter to see the special-population cases alongside the threshold-boundary cases.

In [ ]:
from collections import Counter

SPECIAL_TAGS = {
    'homeless_shelter_deduction',
    'student_exclusion',
    'boarder_income_proration',
    'migrant_income_averaging',
    'mixed_immigration_status_hh_size_reduction',
    'categorical_eligibility_tanf_ssi',
}

cases = gen.generate(n=30, seed=42)

special = [c for c in cases if any(tag in c.variation_tags for tag in SPECIAL_TAGS)]
boundary = [c for c in cases if not any(tag in c.variation_tags for tag in SPECIAL_TAGS)]

print(f'Total cases: {len(cases)}')
print(f'  Special-population: {len(special)} ({len(special)/len(cases)*100:.0f}%)')
print(f'  Threshold-boundary: {len(boundary)} ({len(boundary)/len(cases)*100:.0f}%)')
print()

print('Special-population case breakdown:')
tag_counts = Counter(
    next(tag for tag in c.variation_tags if tag in SPECIAL_TAGS)
    for c in special
)
for tag, n in tag_counts.items():
    outcome_counts = Counter(c.expected_outcome for c in special if tag in c.variation_tags)
    print(f'  {tag:<45} {n}x  ({dict(outcome_counts)})')

print()
print('Threshold-boundary type breakdown:')
for t_type, n in Counter(c.metadata.get('profile_strategy', '?') for c in boundary).most_common():
    print(f'  {t_type:<35} {n}')